In [21]:
# Setup: run every notebook from the project folder, so raw/, interim/, clean/ and data/ resolve the same way.
import os, sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "README.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "README.md").exists():
    raise SystemExit("README.md not found above " + str(Path.cwd()) + ". Open the notebook from inside the project folder.")
os.chdir(ROOT)
print("Project folder:", ROOT)


Project folder: c:\Users\surya\OneDrive\Desktop\Acads\DA2402_DCV\paperrank


In [22]:
import re
import sys
import time
from datetime import date
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

YEARS = range(2017, 2026)
BASE = "https://www.nirfindia.org"
PAGE = BASE + "/Rankings/{year}/OverallRanking.html"
HEADERS = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64; rv:157.0) Gecko/20100101 Firefox/157.0"}
DELAY_S = 1.5
TODAY = date.today().isoformat()

PARAMS = ["tlr", "rpc", "go", "oi", "pr"]

RAW = Path("raw")
INTERIM = Path("interim")


In [23]:
def to_float(text):
    try:
        return float(str(text).replace(",", "").strip())
    except ValueError:
        return None


In [24]:
def to_int(text):
    m = re.fullmatch(r"\s*(\d+)\s*", text or "")
    return int(m.group(1)) if m else None


In [25]:
def hidden_scores(hidden):
    """Parameter scores from the 'More Details' block of one row, as {param: float}."""
    if hidden is None:
        return {}
    values = [td.get_text(strip=True) for td in hidden.select("tbody tr td")]
    heads = [th.get_text(strip=True).lower() for th in hidden.select("thead th")]
    if heads and len(heads) == len(values):
        named = {}
        for head, value in zip(heads, values):
            for p in PARAMS:
                if head.startswith(p):
                    named[p] = to_float(value)
        if len(named) == len(PARAMS):
            return named

    return {p: to_float(v) for p, v in zip(PARAMS, values)}


In [26]:
def parse_year(html, year):
    soup = BeautifulSoup(html, "html.parser")
    body = soup.select_one("tbody")
    if body is None:
        raise RuntimeError(f"{year}: no <tbody> found - the page layout changed")
    records, skipped = [], 0
    for row in body.find_all("tr", recursive=False):
        tds = row.find_all("td", recursive=False)
        if len(tds) != 6:
            skipped += 1
            continue
        name_cell = tds[1]
        first = name_cell.contents[0]
        name = first.get_text(strip=True) if hasattr(first, "get_text") else str(first).strip()
        scores = hidden_scores(name_cell.find("div", class_="tbl_hidden"))
        records.append({
            "year": year,
            "nirf_id": tds[0].get_text(strip=True),
            "name": name,
            "location": tds[2].get_text(strip=True),
            "rank_page": to_int(tds[5].get_text(strip=True)),
            "cell_3": tds[3].get_text(strip=True),   # kept raw for checking which td is which
            "cell_4": tds[4].get_text(strip=True),
            "cell_5": tds[5].get_text(strip=True),
            **{p: scores.get(p) for p in PARAMS},
        })
    for position, rec in enumerate(records, start=1):
        rec["rank_order"] = position
    if not records:
        raise RuntimeError(f"{year}: table found but no 6-column rows parsed")
    return pd.DataFrame(records), skipped


In [27]:
def main():
    RAW.mkdir(exist_ok=True)
    INTERIM.mkdir(exist_ok=True)
    session = requests.Session()
    time.sleep(DELAY_S)

    frames = []
    for year in YEARS:
        url = PAGE.format(year=year)
        r = session.get(url, headers=HEADERS, timeout=60)
        if r.status_code != 200:
            sys.exit(f"{year}: HTTP {r.status_code} for {url}")
        (RAW / f"nirf_overall_{year}_{TODAY}.html").write_bytes(r.content)
        df, skipped = parse_year(r.text, year)
        print(f"{year}: {len(df)} rows parsed, {skipped} rows skipped")
        frames.append(df)
        time.sleep(DELAY_S)

    df = pd.concat(frames, ignore_index=True)
    df["rank_page"] = pd.to_numeric(df["rank_page"], errors="coerce")
    df["rank"] = df["rank_page"].fillna(df["rank_order"]).astype(int)
    df["score"] = df["rpc"]

    out = INTERIM / f"nirf_overall_scraped_{TODAY}.csv"
    df.to_csv(out, index=False)
    print(f"\nsaved {len(df)} rows -> {out}")

    print("\nCheck the cells once by eye (is cell_5 really the rank?):")
    print(df[["year", "name", "cell_3", "cell_4", "cell_5", "rank_page", "rank_order"]].head(3).to_string(index=False))

    if df["rank_page"].isna().any():
        print(f"\nWARNING: rank cell is not a number for {df['rank_page'].isna().sum()} rows; row order was used "
              "for those. Open a saved HTML file and check which <td> holds the rank.")
    mismatch = df[df["rank_page"].notna() & (df["rank_page"] != df["rank_order"])]
    print(f"\nrows where the page rank differs from row order (ties): {len(mismatch)}")
    if len(mismatch):
        print(mismatch[["year", "name", "rank_page", "rank_order"]].to_string(index=False))
    print("\nrows per year:")
    print(df.groupby("year").size().to_string())


In [28]:
sys.argv = ["notebook"] + []
main()


2017: 100 rows parsed, 0 rows skipped
2018: 100 rows parsed, 0 rows skipped
2019: 100 rows parsed, 0 rows skipped
2020: 101 rows parsed, 0 rows skipped
2021: 100 rows parsed, 0 rows skipped
2022: 100 rows parsed, 0 rows skipped
2023: 100 rows parsed, 0 rows skipped
2024: 100 rows parsed, 0 rows skipped
2025: 100 rows parsed, 0 rows skipped

saved 901 rows -> interim\nirf_overall_scraped_2026-11-04.csv

Check the cells once by eye (is cell_5 really the rank?):
 year                                  name      cell_3 cell_4 cell_5  rank_page  rank_order
 2017 Indian Institute of Science Bangalore   Karnataka  83.28      1          1           1
 2017 Indian Institute of Technology Madras  Tamil Nadu  73.97      2          2           2
 2017 Indian Institute of Technology Bombay Maharashtra  71.78      3          3           3

rows where the page rank differs from row order (ties): 22
 year                                                                           name  rank_page  rank_or

In [29]:
import pandas as pd
print("Raw files:")
for f in sorted(Path("raw").glob("*")):
    if f.is_file():
        print(f"  {f.name:45s} {f.stat().st_size / 1024:8.1f} KB")
scraped = sorted(Path("interim").glob("nirf_overall_scraped_*.csv"))
if scraped:
    df = pd.read_csv(scraped[-1])
    print("\nParsed table:", scraped[-1].name, df.shape)
    print(df.groupby("year").size().to_string())


Raw files:
  nirf_overall_2017_2026-10-31.html                602.0 KB
  nirf_overall_2017_2026-11-02.html                602.0 KB
  nirf_overall_2017_2026-11-04.html                602.0 KB
  nirf_overall_2018_2026-10-31.html                636.3 KB
  nirf_overall_2018_2026-11-02.html                636.3 KB
  nirf_overall_2018_2026-11-04.html                636.3 KB
  nirf_overall_2019_2026-10-31.html                119.8 KB
  nirf_overall_2019_2026-11-02.html                119.8 KB
  nirf_overall_2019_2026-11-04.html                119.8 KB
  nirf_overall_2020_2026-10-31.html                 95.8 KB
  nirf_overall_2020_2026-11-02.html                 95.8 KB
  nirf_overall_2020_2026-11-04.html                 95.8 KB
  nirf_overall_2021_2026-10-31.html                109.2 KB
  nirf_overall_2021_2026-11-02.html                109.2 KB
  nirf_overall_2021_2026-11-04.html                109.2 KB
  nirf_overall_2022_2026-10-31.html                109.3 KB
  nirf_overall_2022_2026-11-0